# 13. The LEFT JOIN that a WHERE clause quietly undoes

**Track:** sql | **Difficulty:** easy | **Tags:** left join, on vs where, count, coalesce

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("013-left-join-where-filter")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "customers": """
UEFSMRUEFYAFFd4CTBVQFQASAADAAgjpAwAFAQDqDQgA6w0IAOwNCADtDQgA7g0IAO8NCADwDQgA8Q0IAPINCADzDQgA9A0IAPUN
CAD2DQgA9w0IAPgNCAD5DQgA+g0IAPsNCAD8DQgA/Q0IAP4NCAD/DQgEAAQFtwQAAQ0IAAINCAADDQgABA0IAAUNCAAGDQgABw0I
AAgNCAAJDQgACg0IAAsNCAAMDQgADQ0IAA4NCDwPBAAAAAAAABAEAAAAAAAAFQAVTBVQLBVQFRAVBhUGHBgIEAQAAAAAAAAYCOkD
AAAAAAAAFgAoCBAEAAAAAAAAGAjpAwAAAAAAABERAAAAJpQCAAAAUAEGC0AgDERhHEiiLEzjPFAkTVRlXVimbVznfWAojmRpnhUE
FZQJFa4ETBVQFQASAADKBFwMAAAAQWFyYXYgU2hhcm1hCwAAAERpeWENDwEfEEthYmlyHRAMTWVlci4gABBSb2hhbg0gAA0BTwhu
YW4VQQERFFZpa3JhbQ0iAWEISXNoLkEADEFyanURQQEfBFNhFWAACgFQBZ8QSXllcgkRnQENARsJmxEOCZkRDgmXAQ4BVA2VEQ8N
kwEPAVUFkQENAVUJjwEOARsFjQENBRsFiwxLaGFuARslKAENARsJixEOCYsRDgmLAQ4dixEPDYsBDwFVBYsBDQFVCYsBDgEbBYsB
DQVGBYsQUGF0ZWwBKgWMBQ4BHQmNFQ8JjhUPCY8FDyHgLRsVEA2RBRABWwWSBQ4BWwmTBQ80CgAAAFNhcmEgUGF0ZWwVABVMFVAs
FVAVEBUGFQYcNgAoDVZpa3JhbSBTaGFybWEYCkFhcmF2IEl5ZXIREQAAACaUAgAAAFABBgtAIAxEYRxIoixM4zxQJE1UZV1Ypm1c
531gKI5kaZ4VBBWAARWCAUwVDBUAEgAAQHQFAAAARGVsaGkEAAAAUHVuZQkAAABIeWRlcmFiYWQBDXRCZW5nYWx1cnUGAAAATXVt
YmFpBwAAAENoZW5uYWkVABUuFTIsFVAVEBUGFQYcNgAoBFB1bmUYCUJlbmdhbHVydRERAAAAF1gCAAAAUAEDC0igbQjAsEk2MhwA
dqUyVBUEGUw1ABgGc2NoZW1hFQYAFQQlAhgLY3VzdG9tZXJfaWQAFQwlAhgEbmFtZSUATBwAAAAVDCUCGARjaXR5JQBMHAAAABZQ
GRwZPCYAHBUEGTUABhAZGAtjdXN0b21lcl9pZBUCFlAW6gYWzAQmhgMmCBwYCBAEAAAAAAAAGAjpAwAAAAAAABYAKAgQBAAAAAAA
ABgI6QMAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAUAAAACYAHBUMGTUABhAZGARuYW1lFQIWUBbkChaCBiaiCSbUBBw2
ACgNVmlrcmFtIFNoYXJtYRgKQWFyYXYgSXllchERABksFQQVABUCABUAFRAVAgA8FtQGGQYZJgBQAAAAJgAcFQwZNQAGEBkYBGNp
dHkVAhZQFp4CFqQCJvgLJtYKHDYAKARQdW5lGAlCZW5nYWx1cnUREQAZLBUEFQAVAgAVABUQFQIAPBb0AxkGGSYAUAAAABbsExZQ
JggW8gwAGRwYDEFSUk9XOnNjaGVtYRi4Ai8vLy8vK0FBQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNB
QUlBQUFBQkFBSUFBQUFCQUFBQUFNQUFBQndBQUFBTUFBQUFBUUFBQUNzLy8vL0FBQUJCUkFBQUFBWUFBQUFCQUFBQUFBQUFBQUVB
QUFBWTJsMGVRQUFBQURZLy8vLzFQLy8vd0FBQVFVUUFBQUFIQUFBQUFRQUFBQUFBQUFBQkFBQUFHNWhiV1VBQUFBQUJBQUVBQVFB
QUFBUUFCUUFDQUFHQUFjQURBQUFBQkFBRUFBQUFBQUFBUUlRQUFBQUpBQUFBQVFBQUFBQUFBQUFDd0FBQUdOMWMzUnZiV1Z5WDJs
a0FBZ0FEQUFJQUFjQUNBQUFBQUFBQUFGQUFBQUFBQUFBQUE9PQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4xGTwc
AAAcAAAcAAAA8wIAAFBBUjE=
""",
    "orders": """
UEFSMRUEFYAVFdoKTBXQAhUAEgAAwAoIUcMABQEAUg0IAFMNCABUDQgAVQ0IAFYNCABXDQgAWA0IAFkNCABaDQgAWw0IAFwNCABd
DQgAXg0IAF8NCABgDQgAYQ0IAGINCABjDQgAZA0IAGUNCABmDQgAZw0IAGgNCABpDQgAag0IAGsNCABsDQgAbQ0IAG4NCABvDQgA
cA0IAHENCAByDQgAcw0IAHQNCAB1DQgAdg0IAHcNCAB4DQgAeQ0IAHoNCAB7DQgAfA0IAH0NCAB+DQgAfw0IAIANCACBDQgAgg0I
AIMNCACEDQgAhQ0IAIYNCACHDQgAiA0IAIkNCACKDQgAiw0IAIwNCACNDQgAjg0IAI8NCACQDQgAkQ0IAJINCACTDQgAlA0IAJUN
CACWDQgAlw0IAJgNCACZDQgAmg0IAJsNCACcDQgAnQ0IAJ4NCACfDQgAoA0IAKENCACiDQgAow0IAKQNCAClDQgApg0IAKcNCACo
DQgAqQ0IAKoNCACrDQgArA0IAK0NCACuDQgArw0IALANCACxDQgAsg0IALMNCAC0DQgAtQ0IALYNCAC3DQgAuA0IALkNCAC6DQgA
uw0IALwNCAC9DQgAvg0IAL8NCADADQgAwQ0IAMINCADDDQgAxA0IAMUNCADGDQgAxw0IAMgNCADJDQgAyg0IAMsNCADMDQgAzQ0I
AM4NCADPDQgA0A0IANENCADSDQgA0w0IANQNCADVDQgA1g0IANcNCADYDQgA2Q0IANoNCADbDQgA3A0IAN0NCADeDQgA3w0IAOAN
CADhDQgA4g0IAOMNCADkDQgA5Q0IAOYNCADnDQgA6A0IAOkNCADqDQgA6w0IAOwNCADtDQgA7g0IAO8NCADwDQgA8Q0IAPINCADz
DQgA9A0IAPUNCAD2DQg898MAAAAAAAD4wwAAAAAAABUAFeICFeoCLBXQAhUQFQYVBhwYCPjDAAAAAAAAGAhRwwAAAAAAABYAKAj4
wwAAAAAAABgIUcMAAAAAAAAREQAAALEB8LADAAAA0AIBCCsAAQIDBAUGBwgJCgsMDQ4PEBESExQVFhcYGRobHB0eHyAhIiMkJSYn
KCkqKywtLi8wMTIzNDU2Nzg5Ojs8PT4/QEFCQ0RFRkdISUpLTE1OT1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFy
c3R1dnd4eXp7fH1+f4CBgoOEhYaHiImKi4yNjo+QkZKTlJWWl5iZmpucnZ6foKGio6SlpqcVBBXQBBXIAkwVShUAEgAAqAII6wMA
BQEA/g0IAPANCADpDQgA8Q0IBAYEBScEAPoNEADvDQgA9A0IAAQNIAAFDQgA+A0YAPUNCADsDQgA8w0IAP8NCAD3DQgACQ04APYN
EADyDQgABw0YAAANCAADDQgA+w0gAPkNCAAKDRgA6g0QAAsNEAACDQgEDycJwADtDSAA/A0IAAwNIAD9DRAA7g0IPAgEAAAAAAAA
AQQAAAAAAAAVABWOAhWWAiwV0AIVEBUGFQYcGAgPJwAAAAAAABgI6QMAAAAAAAAWACgIDycAAAAAAAAYCOkDAAAAAAAAEREAAACH
AfCGAwAAANACAQYrQCAMRGEIB1IkysI0zgMRUNRIE1VBknUZENYEWaEshRJNEHBIWjFNC/Egm6UljcNdCvBITIdIjHZghIcVWyZM
XuIMxAcZYIgFj8AZw9MYWCMmmDQy2FMoItUJhcZhwSdIgHgdi3Y04mBcByVCA8AYDYA8JEhABMUwh+KJzOFJFQQVuAQVwgRMFY4B
FQASAACcAvQbAchOAADWTgAA2U4AAMNOAADJTgAAoU4AAKROAACsTgAA4k4AAOROAADATgAA7E4AAO1OAADcTgAAxU4AAOlOAACY
TgAAtE4AAKZOAACnTgAAv04AAOpOAAC+TgAA6E4AAO5OAACpTgAAy04AAKhOAACtTgAA004AAOBOAAC5TgAAwk4AALNOAADSTgAA
404AAOVOAADETgAAz04AAKBOAACwTgAAq04AAL1OAACiTgAAr04AALVOAADeTgAAxk4AAMdOAADbTgAA4U4AAKNOAACdTgAA2E4A
ANBOAADrTgAA1E4AAJ9OAAC8TgAAnk4AAMxOAADvTgAA8E4AAN9OAADdTgAAt04AALtOAAC2TgAAnE4AAJpOAADKTgAAFQAVuAIV
wAIsFdACFRAVBhUGHBgE8E4AABgEmE4AABYAKATwTgAAGASYTgAAEREAAACcAfCbAwAAANACAQcrgIBgQCgYDoiEYsEQNBwPSEQy
iRwoFYviwsBkNJFNgdPxfEChiugzWnxIk5JJckqgUiqNaPW4rFifVsj1gsVgm5VIFjugRITQTASg1UoJW+HmmOFyushpZ/g8eKae
BhHyMVa4X2FzAlJ2H1rOEDKkQjJPIyhU/Ri5WbCDgCkGqdNw0FAdAgiQq9U7BIYfxY9YND4+FQQVShVKTBUGFQASAAAlMAkAAABD
T01QTEVURUQFDUhBTkNFTExFRAcAAABQRU5ESU5HFQAVhAEVigEsFdACFRAVBhUGHDYOKAdQRU5ESU5HGAlDQU5DRUxMRUQREQAA
AELwQRIAAAAUAQPugAEBBf7zQgED/ioBA/4CKxSAkBAIQCCIARAUAAAQAQAAQAQUEJECJABSBCYCJhiiAQAAIZEkAKABABUEFZAN
FZoNTBXQAhUAEgAAyAb0RwMAAASMDwAAAmyCAAANMPMAABAg5wAACK6oAAAGLT8AAAgQEAAABUf0AAAA6fwAAAZsugAAD5XYAAAL
sGIAABAMlwAABSpDAAARtpIAAAypmgAABmcQAAAQtI8AAABw5AAACu5wAAAEs4MAAA1AxgAAAY83AAAGiEQAAAu21AAAC+uPAAAN
lVcAAA8wegAAC16LAAASST8AAAr4+wAAC3NAAAAMK9IAAAbIVAAAED7KAAANVd4AAA/4rAAABytaAAACPdMAAAdyQAAABxiaAAAI
TDcAAAJRXAAABBL4AAAJ4UsAAAQf2wAABabgAAAKrQIAAAiTswAAC/OSAAAB3CwAAArOmgAADugDAAAAyx8AAAnQTgAAAHNuAAAH
xN4AAArHwwAADfULAAAF0UIAAAXKOgAACzL+AAABB3oAAAi1SwAAEUhqAAAEbi4AAAN+6wAADagYAAAKJ5sAAA5p1wAAAPssAAAQ
XzYAAAOySgAABNigAAAGzsYAAAh8qAAAC6uyAAAIvbIAAAdCZgAABpwwAAAMD4AAAA3JsAAACQ0wAAAQjkgAABCY1AAAARQsAAAF
SYQAAA/V6AAABkRMAAAMJJcAAAMqJwAABcDaAAARyn4AAAmGSAAAD3N3AAASQwAAAARl+gAAAwHrAAAA8f4AAA6y4wAACa/iAAAS
LYQAAAI0cwAAEZILAAAGX6QAABG7pgAAEWkHAAADVLsAAAFUbgAADZ9OAAAQ14YAAAIHIwAAEH4SAAAAnsoAAAyO2AAACyQmAAAQ
MYIAABH8TAAACzDYAAANWS8AAAcOowAADcISAAAL7O4AAA8dugAACFw8AAACeDoAAAuSgAAAC+vzAAAPj/sAAAnHUgAAAQWGAAAI
bdAAAAx+DAAACSyiAAAI0A4AAA81WwAABzYXAAAAjvcAAAUUZAAABlanAAAKdrYAAA4u3AAAEZF2AAAQc7cAAAyS8gAAALvkAAAH
qIwAAApqAwAABmL2AAAQ51cAAAmHpgAADiRPAAAI5e4AAA4QMgAACRmvAAALpdYAAAf4CgAABhEgAAAFHloAAANK9wAADbNrAAAE
7oAAAAW24wAADGjCAAAPCysAAA85EgAAAtEYAAAHtXAVABXiAhXqAiwV0AIVEBUGFQYcGAUAABJJPxgFAAAAcOQWACgFAAASST8Y
BQAAAHDkEREAAACxAfCwAwAAANACAQgrAAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEy
MzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6e3x9
fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6ChoqOkpaanFQQZbDUAGAZzY2hlbWEVCgAVBCUCGAhvcmRlcl9pZAAV
BCUCGAtjdXN0b21lcl9pZAAVAiUCGApvcmRlcl9kYXRlJQxMbAAAABUMJQIYBnN0YXR1cyUATBwAAAAVDhUKFQIYBmFtb3VudCUK
FQQVFCxcFQQVFAAAABbQAhkcGVwmABwVBBk1AAYQGRgIb3JkZXJfaWQVAhbQAhaIGRbqDiaECyYIHBgI+MMAAAAAAAAYCFHDAAAA
AAAAFgAoCPjDAAAAAAAAGAhRwwAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgDQAgAAACYAHBUEGTUABhAZGAtjdXN0b21l
cl9pZBUCFtACFoIIFoIGJtoRJvIOHBgIDycAAAAAAAAYCOkDAAAAAAAAFgAoCA8nAAAAAAAAGAjpAwAAAAAAABERABksFQQVABUC
ABUAFRAVAgA8KQYZJgDQAgAAACYAHBUCGTUABhAZGApvcmRlcl9kYXRlFQIW0AIW9gcWiAgm2Bkm9BQcGATwTgAAGASYTgAAFgAo
BPBOAAAYBJhOAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA0AIAAAAmABwVDBk1AAYQGRgGc3RhdHVzFQIW0AIWxgIWzAIm4h0m
/BwcNg4oB1BFTkRJTkcYCUNBTkNFTExFRBERABksFQQVABUCABUAFRAVAgA8FvIVGQYZJg7CAgAAACYAHBUOGTUABhAZGAZhbW91
bnQVAhbQAhaAERaSESaELSbIHxwYBQAAEkk/GAUAAABw5BYAKAUAABJJPxgFAAAAcOQREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA
0AIAAAAWxjwW0AImCBbSMAAZHBgMQVJST1c6c2NoZW1hGOADLy8vLy8yQUJBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFB
QkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQVVBQUFEMEFBQUFyQUFBQUhBQUFBQkFBQUFBQkFBQUFERC8vLzhBQUFFSEVB
QUFBQ0FBQUFBRUFBQUFBQUFBQUFZQUFBQmhiVzkxYm5RQUFBZ0FEQUFFQUFnQUNBQUFBQW9BQUFBQ0FBQUFhUC8vL3dBQUFRVVFB
QUFBSEFBQUFBUUFBQUFBQUFBQUJnQUFBSE4wWVhSMWN3QUFCQUFFQUFRQUFBQ1UvLy8vQUFBQkNCQUFBQUFrQUFBQUJBQUFBQUFB
QUFBS0FBQUFiM0prWlhKZlpHRjBaUUFBQUFBR0FBZ0FCZ0FHQUFBQUFBQUFBTXovLy84QUFBRUNFQUFBQUJ3QUFBQUVBQUFBQUFB
QUFBc0FBQUJqZFhOMGIyMWxjbDlwWkFDOC8vLy9BQUFBQVVBQUFBQVFBQlFBQ0FBR0FBY0FEQUFBQUJBQUVBQUFBQUFBQVFJUUFB
QUFKQUFBQUFRQUFBQUFBQUFBQ0FBQUFHOXlaR1Z5WDJsa0FBQUFBQWdBREFBSUFBY0FDQUFBQUFBQUFBRkFBQUFBABggcGFycXVl
dC1jcHAtYXJyb3cgdmVyc2lvbiAyNS4wLjEZXBwAABwAABwAABwAABwAAACyBAAAUEFSMQ==
""",
}

_data_dir = pathlib.Path("data") / "013-left-join-where-filter"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

customers = spark.read.parquet((_data_dir / "customers.parquet").resolve().as_posix())
customers.createOrReplaceTempView("customers")
orders = spark.read.parquet((_data_dir / "orders.parquet").resolve().as_posix())
orders.createOrReplaceTempView("orders")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'completed_amount': '5d2c697ef6fd97595c06ceb80c0f037c99ee5bfc8192c785e928161ecc6c9370',
                         'completed_orders': '835c226ffa40d7433ded91c041eabd3aef45176eddc849308c5173a129ccdbf9',
                         'customer_id': '0d6c61500b1ab4aa4fa607953ad73f4c32116ae9c2808974e5c5fab740bc2fbf',
                         'name': '387126943c5ab543fbad7b2d0d0c6e6b5c4d42c190aff722cc5fbe1c753f3ed9'},
 'columns': [['completed_amount', 'decimal(12,2)'],
             ['completed_orders', 'bigint'],
             ['customer_id', 'bigint'],
             ['name', 'string']],
 'fingerprint': 'c17a703628356660c50b1113ff16531588701862b5e519601b670442759d43e3',
 'order_matters': False,
 'row_count': 40}

check = make_check(EXPECTED)

## The situation

Account managers get a monthly list of all their customers with how much each one
bought. Customers who bought nothing matter most on that list, because those are the
ones to call.

Last month's list was short. Every customer on it had bought something.

## Input tables

### `customers`

One row per customer.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | Unique per customer. |
| `name` | STRING | Customer name. |
| `city` | STRING | Customer city. |

### `orders`

One row per order.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | Unique per order. |
| `customer_id` | BIGINT | The customer who ordered. A few orders belong to customers that are not in `customers`. |
| `order_date` | DATE | Date of the order. |
| `status` | STRING | `COMPLETED`, `CANCELLED` or `PENDING`. Can be NULL. |
| `amount` | DECIMAL(10,2) | Order value in rupees. |

## Requirement

Return every customer in `customers` with their completed orders of March 2025.

- Only orders with `status` = `COMPLETED` and an `order_date` from 1 March 2025 to
  31 March 2025 (both included) count.
- `completed_orders` is the number of such orders. `completed_amount` is their total.
- A customer with no such order still appears, with `completed_orders` = 0 and
  `completed_amount` = 0.00.
- Orders of customers that are not in `customers` are ignored.

## Expected output

One row per customer in `customers`.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | |
| `name` | STRING | |
| `completed_orders` | BIGINT | 0 if none. |
| `completed_amount` | DECIMAL(12,2) | 0.00 if none. |

In [ ]:
customers.show(5)
orders.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `customers`, `orders`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)